In [ ]:
"""
Cubes Usage Examples with IBRD Data
This shows how to use Cubes once it's properly installed and configured
"""
#Create Workspace
from cubes import Workspace

workspace=Workspace(load_base_model=False)
workspace.register_default_store("sql", url="sqlite:///../data/irbd_db.sqlite")
workspace.import_model("../models/model_example.json")

browser=workspace.browser("irbd_balance")
cube=workspace.cube("irbd_balance")

In [5]:
# ============================================================================
# EXAMPLE 1: Simple Aggregation (Total)
# ============================================================================

result=browser.aggregate()

print(f"total amount: ${result.summary['amount_sum']}M")

total amount: $1116860M


In [ ]:
# ============================================================================
# EXAMPLE 2: Drilldown by One Dimension (YEAR)
# ============================================================================

result=browser.aggregate(drilldown=["year"])

for cell in result.cells:
    year=cell['year']
    amount=cell['amount_sum']
    print(f"year: {year}, total amount: ${amount}M")

year: 2009, total amount: $550840M
year: 2010, total amount: $566020M


In [ ]:
print("\nTest 3: By Category")
# Drill down using the hierarchical 'account' dimension

result=browser.aggregate(drilldown=['account'])

for cell in result.cells:
    category=cell['account.category']
    amount=cell['amount_sum']
    print(f"category: {category}, total_amount: ${amount}M")


Test 3: By Category
category: Assets, total_amount: $558430M
category: Equity, total_amount: $77592M
category: Liabilities, total_amount: $480838M


In [34]:
print("\nTest 4: By Category and Year (FIXED!)")

result=browser.aggregate(drilldown=['account.category','year'])

current_cat=None
for cell in result.cells:
    category=cell['account.category']
    if category != current_cat:
        print(f"\n{category}:")
        current_cat=category
    
    year=cell['year']
    amount=cell['amount_sum']
    print(f"year: {year}, amount: ${amount}M")


Test 4: By Category and Year (FIXED!)

Assets:
year: 2009, amount: $275420M
year: 2010, amount: $283010M

Equity:
year: 2009, amount: $40037M
year: 2010, amount: $37555M

Liabilities:
year: 2009, amount: $235383M
year: 2010, amount: $245455M


In [ ]:
from cubes import PointCut, Cell

# ============================================================================
# EXAMPLE 5: Multiple Slices (Filter to Assets in 2010)
# ============================================================================

cuts=[
PointCut("account",["Assets"]),
PointCut("year",[2010])
]

cell=Cell(cube,cuts=cuts)

result=browser.aggregate(cell,drilldown=["account.subcategory"])
print("Assets(2010) Subcategories:\n")
for row in result.cells:
    subcategory=row['account.subcategory']
    amount=row['amount_sum']
    print(f"{subcategory}, year: {year}, amount: ${amount}M")

Assets(2010) Subcategories:

Assets, Derivative Assets, year: 2010, amount: $121626M
Assets, Due from Banks, year: 2010, amount: $1803M
Assets, Investments, year: 2010, amount: $36012M
Assets, Loans Outstanding, year: 2010, amount: $118104M
Assets, Nonnegotiable, year: 2010, amount: $1123M
Assets, Other Assets, year: 2010, amount: $3071M
Assets, Other Receivables, year: 2010, amount: $811M
Assets, Receivables, year: 2010, amount: $171M
Assets, Securities, year: 2010, amount: $289M


In [44]:
#First 5 asset records from 2010

cuts=[
    PointCut("account",["Assets"]),
    PointCut("year",[2010])
]

cell=Cell(cube,cuts=cuts)

facts=browser.facts(cell)
print("First 5 Individual assets records(2010)")
for i, row in enumerate(facts,0):
    if i == 5:
        break
    category=row.get('account.category')
    year=row.get('year')
    subcategory=row.get('account.subcategory')
    line_item=row.get('account.line_item')
    amount=row.get('amount')
    print(f"{i+1}. {category}, {subcategory}, {line_item}, {year}: ${amount}M")
    

First 5 Individual assets records(2010)
1. Assets, Due from Banks, Unrestricted currencies, 2010: $1581M
2. Assets, Due from Banks, Currencies subject to restriction, 2010: $222M
3. Assets, Investments, Trading, 2010: $36012M
4. Assets, Securities, Securities purchased under resale agreements, 2010: $289M
5. Assets, Nonnegotiable, Nonnegotiable, nonintrest-bearing demand obligations on account of subscribed capital, 2010: $1123M


In [46]:
# ============================================================================
# EXAMPLE 8: Complex Query with Native Sorting & Filtering
# ============================================================================

print("Find: What are the biggest liability items in 2010?")

cuts=[
    PointCut("account", ["Liabilities"]),
    PointCut("year",[2010])
]

cell=Cell(cube,cuts=cuts)

facts=browser.facts(cell)

sorted_facts=sorted(
    facts,
    key=lambda x: x.get('amount',0),
    reverse=True
)
print("5 biggest liabilities items(2010):\n")
for i, fact in enumerate(sorted_facts, 0):
    if i == 5:
        break
    category=fact.get('account.category')
    subcategory=fact.get('account.subcategory')
    line_item=fact.get('account.line_item')
    year=fact.get('year')
    amount=fact.get('amount',0)
    print(f"{i+1}. {category}, {subcategory},{line_item},{year}, amount: ${amount}M")

Find: What are the biggest liability items in 2010?
5 biggest liabilities items(2010):

1. Liabilities, Borrowings,All,2010, amount: $128577M
2. Liabilities, Derivative Liabilities,Borrowings,2010, amount: $78655M
3. Liabilities, Derivative Liabilities,Client Operations,2010, amount: $17623M
4. Liabilities, Derivative Liabilities,Investments,2010, amount: $13360M
5. Liabilities, Other Liabilities,Accounts payable and misc liabilities,2010, amount: $2793M
